# Geocoder Comparison: SQLite/Ruby vs DuckDB/Python

This notebook compares the geocoding results and performance between:
- **Original**: Ruby implementation with SQLite3
- **New**: Python implementation with DuckDB

## Setup

First, ensure you have both databases available:
- Ruby/SQLite database (typically `geocoder.db`)
- Python/DuckDB database (typically `geocoder.duckdb`)

In [ ]:
# Import required libraries
import sys
import time
import subprocess
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Python geocoder
from geocoder_us import Database, Address

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Test Addresses

Define a set of test addresses covering different scenarios:
- Street addresses
- City/State queries
- ZIP codes
- PO Boxes
- Intersections

In [ ]:
test_addresses = [
    # Street addresses
    "1600 Pennsylvania Ave NW, Washington DC 20500",
    "350 5th Ave, New York NY 10118",
    "1 Infinite Loop, Cupertino CA 95014",
    "1 Microsoft Way, Redmond WA 98052",
    "1600 Amphitheatre Parkway, Mountain View CA 94043",
    
    # City/State
    "Seattle, WA",
    "Portland, OR",
    "San Francisco, CA",
    "New York, NY",
    "Chicago, IL",
    
    # ZIP codes
    "10001",
    "90210",
    "20500",
    "60601",
    
    # Varied formats
    "123 Main Street, Springfield IL",
    "456 Oak Avenue, Portland OR 97201",
    "789 Pine Road, Denver CO 80202",
]

print(f"Testing with {len(test_addresses)} addresses")

## Python/DuckDB Geocoding

Test the Python implementation with DuckDB.

In [ ]:
# Set path to your DuckDB database
duckdb_path = "geocoder.duckdb"

# Check if database exists
if not Path(duckdb_path).exists():
    print(f"⚠️  Database not found: {duckdb_path}")
    print("Please create a database first using:")
    print("  python tools/tiger_import_duckdb.py geocoder.duckdb /path/to/tiger/")
else:
    print(f"✓ Database found: {duckdb_path}")

In [ ]:
def geocode_python(address_str: str, db_path: str) -> dict:
    """Geocode using Python/DuckDB implementation."""
    start = time.time()
    
    try:
        with Database(db_path, debug=False) as db:
            results = db.geocode(address_str)
            
        elapsed = time.time() - start
        
        if results:
            top_result = results[0]
            return {
                'address': address_str,
                'lat': top_result.get('lat', None),
                'lon': top_result.get('lon', None),
                'score': top_result.get('score', None),
                'precision': top_result.get('precision', None),
                'city': top_result.get('city', ''),
                'state': top_result.get('state', ''),
                'zip': top_result.get('zip', ''),
                'elapsed_ms': elapsed * 1000,
                'found': True
            }
    except Exception as e:
        print(f"Error geocoding '{address_str}': {e}")
    
    return {
        'address': address_str,
        'lat': None,
        'lon': None,
        'score': None,
        'precision': None,
        'city': '',
        'state': '',
        'zip': '',
        'elapsed_ms': (time.time() - start) * 1000,
        'found': False
    }

# Test with Python/DuckDB (if database exists)
python_results = []
if Path(duckdb_path).exists():
    print("\nGeocoding with Python/DuckDB...")
    for addr in test_addresses:
        result = geocode_python(addr, duckdb_path)
        python_results.append(result)
        status = "✓" if result['found'] else "✗"
        print(f"{status} {addr[:50]:50s} | {result['elapsed_ms']:6.2f}ms")
    
    # Create DataFrame
    python_df = pd.DataFrame(python_results)
    print(f"\n📊 Python/DuckDB Results:")
    print(f"   Found: {python_df['found'].sum()}/{len(python_df)}")
    print(f"   Avg time: {python_df['elapsed_ms'].mean():.2f}ms")
else:
    print("⚠️  Skipping Python/DuckDB tests (database not found)")

## Ruby/SQLite Geocoding

Test the Ruby implementation with SQLite (if available).

In [ ]:
def geocode_ruby(address_str: str, db_path: str = None) -> dict:
    """Geocode using Ruby/SQLite implementation."""
    start = time.time()
    
    # Check if Ruby and geocoder-us gem are available
    try:
        # Create a simple Ruby script to geocode
        ruby_script = f"""
require 'geocoder/us'
require 'json'

db = Geocoder::US::Database.new('{db_path}')
results = db.geocode('{address_str}')

if results && results.length > 0
  result = results[0]
  puts JSON.generate({{
    lat: result[:lat],
    lon: result[:lon],
    score: result[:score],
    city: result[:city],
    state: result[:state],
    zip: result[:zip],
    found: true
  }})
else
  puts JSON.generate({{found: false}})
end
"""
        
        # Run Ruby script
        result = subprocess.run(
            ['ruby', '-e', ruby_script],
            capture_output=True,
            text=True,
            timeout=10
        )
        
        elapsed = time.time() - start
        
        if result.returncode == 0 and result.stdout:
            data = json.loads(result.stdout)
            data['address'] = address_str
            data['elapsed_ms'] = elapsed * 1000
            return data
        else:
            print(f"Ruby error: {result.stderr}")
    
    except FileNotFoundError:
        print("Ruby not found - skipping Ruby tests")
        return None
    except Exception as e:
        print(f"Error running Ruby geocoder: {e}")
    
    return {
        'address': address_str,
        'lat': None,
        'lon': None,
        'score': None,
        'city': '',
        'state': '',
        'zip': '',
        'elapsed_ms': (time.time() - start) * 1000,
        'found': False
    }

# Test with Ruby/SQLite (if available)
ruby_db_path = "geocoder.db"  # Adjust if needed
ruby_results = []

print("\nTesting Ruby geocoder availability...")
test_result = geocode_ruby(test_addresses[0], ruby_db_path)

if test_result is None:
    print("⚠️  Ruby/SQLite geocoding not available")
    print("   This comparison will only show Python/DuckDB results")
elif Path(ruby_db_path).exists():
    print(f"✓ Ruby geocoder available with database: {ruby_db_path}")
    print("\nGeocoding with Ruby/SQLite...")
    for addr in test_addresses:
        result = geocode_ruby(addr, ruby_db_path)
        if result:
            ruby_results.append(result)
            status = "✓" if result['found'] else "✗"
            print(f"{status} {addr[:50]:50s} | {result['elapsed_ms']:6.2f}ms")
    
    if ruby_results:
        ruby_df = pd.DataFrame(ruby_results)
        print(f"\n📊 Ruby/SQLite Results:")
        print(f"   Found: {ruby_df['found'].sum()}/{len(ruby_df)}")
        print(f"   Avg time: {ruby_df['elapsed_ms'].mean():.2f}ms")
else:
    print(f"⚠️  Ruby database not found: {ruby_db_path}")

## Performance Comparison

Compare geocoding performance between implementations.

In [ ]:
# Plot performance comparison
if python_results and ruby_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Average time comparison
    avg_times = pd.DataFrame({
        'Implementation': ['Ruby/SQLite', 'Python/DuckDB'],
        'Average Time (ms)': [
            ruby_df['elapsed_ms'].mean(),
            python_df['elapsed_ms'].mean()
        ]
    })
    
    axes[0].bar(avg_times['Implementation'], avg_times['Average Time (ms)'])
    axes[0].set_ylabel('Time (ms)')
    axes[0].set_title('Average Geocoding Time')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Success rate comparison
    success_rates = pd.DataFrame({
        'Implementation': ['Ruby/SQLite', 'Python/DuckDB'],
        'Success Rate (%)': [
            ruby_df['found'].sum() / len(ruby_df) * 100,
            python_df['found'].sum() / len(python_df) * 100
        ]
    })
    
    axes[1].bar(success_rates['Implementation'], success_rates['Success Rate (%)'])
    axes[1].set_ylabel('Success Rate (%)')
    axes[1].set_title('Geocoding Success Rate')
    axes[1].set_ylim([0, 100])
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate speedup
    speedup = ruby_df['elapsed_ms'].mean() / python_df['elapsed_ms'].mean()
    print(f"\n⚡ Speedup: Python/DuckDB is {speedup:.2f}x faster than Ruby/SQLite")
    
elif python_results:
    # Show Python-only results
    fig, ax = plt.subplots(figsize=(10, 5))
    python_df['elapsed_ms'].plot(kind='bar', ax=ax)
    ax.set_xlabel('Address Index')
    ax.set_ylabel('Time (ms)')
    ax.set_title('Python/DuckDB Geocoding Time per Address')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Statistics:")
    print(f"   Min time: {python_df['elapsed_ms'].min():.2f}ms")
    print(f"   Max time: {python_df['elapsed_ms'].max():.2f}ms")
    print(f"   Median time: {python_df['elapsed_ms'].median():.2f}ms")

## Result Accuracy Comparison

Compare the actual geocoding results to ensure consistency.

In [ ]:
if python_results and ruby_results:
    # Compare coordinates
    comparison = pd.DataFrame({
        'Address': [r['address'][:40] + '...' if len(r['address']) > 40 else r['address'] 
                   for r in python_results],
        'Python Lat': [r['lat'] for r in python_results],
        'Ruby Lat': [r['lat'] for r in ruby_results],
        'Python Lon': [r['lon'] for r in python_results],
        'Ruby Lon': [r['lon'] for r in ruby_results],
        'Python Found': [r['found'] for r in python_results],
        'Ruby Found': [r['found'] for r in ruby_results],
    })
    
    # Calculate coordinate differences (for found addresses)
    comparison['Lat Diff'] = abs(comparison['Python Lat'] - comparison['Ruby Lat'])
    comparison['Lon Diff'] = abs(comparison['Python Lon'] - comparison['Ruby Lon'])
    
    print("\n📍 Coordinate Comparison (first 10 addresses):")
    print(comparison[['Address', 'Python Found', 'Ruby Found', 'Lat Diff', 'Lon Diff']].head(10))
    
    # Summary statistics
    found_both = comparison[(comparison['Python Found']) & (comparison['Ruby Found'])]
    if len(found_both) > 0:
        print(f"\n📊 Coordinate Difference Statistics (for addresses found by both):")
        print(f"   Addresses found by both: {len(found_both)}")
        print(f"   Avg Lat difference: {found_both['Lat Diff'].mean():.6f}°")
        print(f"   Avg Lon difference: {found_both['Lon Diff'].mean():.6f}°")
        print(f"   Max Lat difference: {found_both['Lat Diff'].max():.6f}°")
        print(f"   Max Lon difference: {found_both['Lon Diff'].max():.6f}°")
elif python_results:
    # Show Python results only
    python_only = pd.DataFrame(python_results)
    print("\n📍 Python/DuckDB Geocoding Results:")
    print(python_only[['address', 'lat', 'lon', 'city', 'state', 'zip', 'found']].head(10))

## Address Parsing Comparison

Compare address parsing between implementations.

In [ ]:
# Test address parsing with Python implementation
print("\n📝 Address Parsing Examples (Python):")
print("=" * 70)

sample_addresses = [
    "1600 Pennsylvania Ave NW, Washington DC 20500",
    "350 5th Ave, New York NY 10118",
    "Main St & 1st Ave, Seattle WA",
    "PO Box 123, Portland OR 97201",
    "Seattle, WA",
]

for addr_str in sample_addresses:
    addr = Address(addr_str)
    print(f"\nInput: {addr_str}")
    print(f"  Number: {addr.number}")
    print(f"  Street: {addr.street[:2] if len(addr.street) > 2 else addr.street}")
    print(f"  City:   {addr.city}")
    print(f"  State:  {addr.state}")
    print(f"  ZIP:    {addr.zip}")
    
    if addr.po_box():
        print("  ✉️  PO Box detected")
    if addr.intersection():
        print("  ✖️  Intersection detected")

## Summary

### Key Findings

1. **Performance**: DuckDB/Python typically provides faster query performance due to:
   - Columnar storage
   - Better parallel query execution
   - Optimized spatial operations

2. **Ease of Use**: Python implementation offers:
   - No C extensions to compile
   - Context manager support
   - Type hints for better IDE support

3. **Compatibility**: Both implementations should produce similar results for the same TIGER/Line data

### Next Steps

- Build a complete TIGER/Line database using `tools/tiger_import_duckdb.py`
- Run comprehensive benchmarks with larger address sets
- Test with production workloads

## Export Results

Save comparison results to CSV for further analysis.

In [ ]:
# Export results
if python_results:
    output_file = 'geocoding_comparison_results.csv'
    
    if ruby_results:
        # Combine both results
        combined = []
        for p, r in zip(python_results, ruby_results):
            combined.append({
                'address': p['address'],
                'python_lat': p['lat'],
                'python_lon': p['lon'],
                'python_found': p['found'],
                'python_time_ms': p['elapsed_ms'],
                'ruby_lat': r['lat'],
                'ruby_lon': r['lon'],
                'ruby_found': r['found'],
                'ruby_time_ms': r['elapsed_ms'],
            })
        df = pd.DataFrame(combined)
    else:
        # Python results only
        df = pd.DataFrame(python_results)
    
    df.to_csv(output_file, index=False)
    print(f"\n💾 Results saved to: {output_file}")